In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the duration and the pitch shift independently. The plot shows the jazz
trio (blue) and the granular output (purple), with dashed gold lines that
follow a few moments of the input to where they land in the output. The audio
card plays the result. Change the grain length too: very short grains turn
the sound buzzy, and very long ones smear its rhythm.

In [ ]:
# hide
# autorun
STRETCH0, STEPS0, GRAIN0 = 2.0, 0, 50.0     # half speed, original pitch

PURPLE = "#6E3B87"
trio = pq.Audio.from_file("../assets/audio-trio.wav").as_mono()
SR = trio.sample_rate
X = np.asarray(trio.samples).reshape(-1)
T_IN = len(X) / SR
T_MAX = 2.0 * T_IN                          # the longest output, at half speed
MARKS = np.arange(1.0, T_IN, 1.0)           # input times we follow to the output


def granular(x, stretch, steps, grain_ms, SR=SR):
    """Granular time stretch and pitch shift. Output grains are Hann windows
    of N_F samples at a hop of N_F / 4 (COLA, with c = 2). Each grain is read
    from the input at its own hop N_F / (4 stretch), and resampled by the
    ratio 2^(steps / 12) on the way in."""
    N_F = int(grain_ms / 1000 * SR) // 4 * 4
    hop_out = N_F // 4
    hop_in = hop_out / stretch
    ratio = 2 ** (steps / 12)
    w = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N_F) / N_F))
    read = np.arange(N_F) * ratio           # where one grain reads, from its start
    n_out = int(len(x) * stretch)
    y = np.zeros(n_out + N_F)
    for k in range(n_out // hop_out):
        p = k * hop_in + read
        if p[-1] >= len(x) - 1:
            break
        i = p.astype(int)
        a = p - i
        y[k * hop_out:k * hop_out + N_F] += w * ((1 - a) * x[i] + a * x[i + 1])
    return y[:n_out] / 2, N_F, hop_in, hop_out


def envelope(x, t0, dt, n_blocks=900):
    """Times and min/max values of x in n_blocks blocks, so peaks survive."""
    step = max(1, len(x) // n_blocks)
    n = len(x) // step * step
    blocks = x[:n].reshape(-1, step)
    t = t0 + (np.arange(n // step) + 0.5) * step * dt
    return np.repeat(t, 2), np.column_stack([blocks.min(1), blocks.max(1)]).ravel()


T_ENV, Y_ENV = envelope(X / np.abs(X).max(), 0.0, 1 / SR)


def marks(stretch, MARKS=MARKS):
    """Dashed connectors from each marked input time to its output time."""
    xs, ys = [], []
    for t in MARKS:
        xs += [t, t * stretch, None]
        ys += [0.15, -0.15, None]
    return xs, ys


def figure():
    fig = go.Figure()
    fig.add_scatter(x=T_ENV, y=1.15 + Y_ENV, mode="lines",
                    line=dict(color=BLUE, width=1))
    fig.add_scatter(x=T_ENV * STRETCH0, y=-1.15 + Y_ENV, mode="lines",
                    line=dict(color=PURPLE, width=1))
    xs, ys = marks(STRETCH0)
    fig.add_scatter(x=xs, y=ys, mode="lines",
                    line=dict(color=GOLD, width=2, dash="dash"))
    fig.add_annotation(x=0, y=2.35, text="input", showarrow=False,
                       xanchor="left", font=dict(color=BLUE))
    fig.add_annotation(x=0, y=0.05, text="output", showarrow=False,
                       xanchor="left", font=dict(color=PURPLE))
    fig.update_xaxes(range=[0, T_MAX], title_text="Time (s)", fixedrange=True)
    fig.update_yaxes(range=[-2.4, 2.45], showticklabels=False, ticks="",
                     fixedrange=True)
    return fig


def controls(fig):
    stretch = widgets.FloatSlider(description="Duration (×)", min=0.5,
                                  max=2.0, step=0.05, value=STRETCH0)
    steps = widgets.IntSlider(description="Pitch shift (steps)", min=-12,
                              max=12, value=STEPS0)
    grain = widgets.FloatSlider(description="Grain length (ms)", min=10.0,
                                max=200.0, step=5.0, value=GRAIN0,
                                readout_format=".0f")
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(stretch, steps, grain, T_ENV=T_ENV, marks=marks, SR=SR,
               T_IN=T_IN, readout=readout):
        xs, ys = marks(stretch)
        with fig.batch_update():
            fig.data[1].x = T_ENV * stretch
            fig.data[2].x, fig.data[2].y = xs, ys
        N_F = int(grain / 1000 * SR) // 4 * 4
        hop_out = N_F // 4
        readout.value = (
            f"<span style='font-size:0.9em'><i>N</i><sub>F</sub> = {N_F:,} "
            f"&nbsp;·&nbsp; extract at <i>N</i><sub>H</sub> = "
            f"{hop_out / stretch:,.0f}, reassemble at <i>N</i><sub>H</sub>&prime; = "
            f"{hop_out:,} samples &nbsp;·&nbsp; {T_IN:.1f} s &rarr; "
            f"{T_IN * stretch:.1f} s &nbsp;·&nbsp; pitch &times;{2 ** (steps / 12):.3f}"
            f"</span>")

    widgets.interactive_output(update, {"stretch": stretch, "steps": steps,
                                        "grain": grain})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(X=X, granular=granular):
        y = granular(X, stretch.value, steps.value, grain.value)[0]
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)
        y[:441] *= np.linspace(0, 1, 441)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (stretch, steps, grain):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([stretch, steps, grain, readout, out, gate])

icm_plotly.show(figure, controls)